In [ ]:
import torch
import cs336_basics

        
print("PyTorch Version:", torch.__version__)
print("Assignment code loaded successfully!")

PyTorch Version: 2.11.0+cu130
Assignment code loaded successfully!


In [ ]:
"""
What are some reasons to prefer training our tokenizer on UTF-8 encoded bytes, rather than
UTF-16 or UTF-32? It may be helpful to compare the output of these encodings for various
input strings.
"""
# 1. Vocab compactness & representation
## UTF-8 variable-length scheme (1 to 4 bytes per character, espeically the standard ASCII set)
## so the base byte value is capped at 256.

test_str_list = ['1', '#', '我去上学校']
for test_str in test_str_list:
    print('utf-8:', test_str.encode('utf-8'))
    print('utf-16:', test_str.encode('utf-16'))

# 2. Compare encodings
## The "Null Byte" padding for utf-32 
## English/Code Bias Prevention: while some Asian characters in utf-8 is slightly longer, it
## drastically smaller for code, markdown, syntax and English. So minimize the total byte length globally.

# 3. Universal Coverage and OOV Miracle
## utf-8 breaks everything down into a close universe of 256 possible bytes, can be written as a sequence 
## of those 256 bytes. The tokenizer will always be able to process it. utf-16, i.e., 2^16 = 65536

# 4. Native Ecosystem Alignment
## Most of the web content use utf-8


utf-8: b'1'
utf-16: b'\xff\xfe1\x00'
utf-8: b'#'
utf-16: b'\xff\xfe#\x00'
utf-8: b'\xe6\x88\x91\xe5\x8e\xbb\xe4\xb8\x8a\xe5\xad\xa6\xe6\xa0\xa1'
utf-16: b'\xff\xfe\x11b\xbbS\nNf[!h'


In [ ]:
# why the following function is wrong:
def decode_utf8_bytes_to_str_wrong(bytestring: bytes):
    return "".join([bytes([b]).decode("utf-8") for b in bytestring])

# The reason is: it treats a single byte as a complete character, while utf-8 is variable-length encoding,
# so each char is 1-4 bytes.

def decode_utf8_bytes_to_str(bytestring: bytes):
    return bytestring.decode("utf-8")

In [12]:
# Give a two-byte sequence that does not decode to any Unicode character(s).
# \x00\x00?
(b'\xff\xff').decode('utf-8')

UnicodeDecodeError: 'utf-8' codec can't decode byte 0xff in position 0: invalid start byte

In [ ]:
# Print nothing visible
print('\x00')

 


In [ ]:
# When chr(0) occurs in text, Python preserves and concatenates the null byte within string operations,
# but printing or processing it in external environments (like C-based libraries or display terminals)
# can result in it appearing invisible, truncating the string, or causing rendering anomalies.
print("this is a test" + chr(0) + "string")

this is a test string


In [ ]:
# Output is "'\\x00'"
'\x00'.__repr__()

"'\\x00'"

In [ ]:
print(chr(0))

 


In [7]:
# Null character
chr(0)

'\x00'

# Subword Tokenization
Tokenize text into bytes results in extremely long input sequences. b'the' will be one token only for subword tokenization. Byte-pair encoding (BPE) - a compression algorithm that iteratively replaces("merge") the most frequent pair of bytes with a single, new unused index. -> Maximize the compression of input sequences.

# BPE Tokenizer
Vocabulary items are bytes or merged sequences of bytes.
    1) OOV handling
    2) manageable input sequence lengths

"Training" a BPE tokenizer: constructing the BPE tokenizer vocabulary.

# BPE Tokenizer Training
1. Vocabulary initialization

The tokenizer vocabulary is a one-to-one mapping from bytestring to integer ID.
For a byte-level tokenizer, initial vocabulary is simply the set of all bytes.
256 possible bytes values -> initial vocab size is 256.

2. Pre-tokenization

Count how often bytes occur next to each other and merging them starting with the most frequent pair of bytes.
-> computationally expensive, as we need to take a full pass each time we merge.

Also, directly merge will result in tokens that differ only in punctuation. dog! vs dog. (completely different token IDs, even though with similar semantic meaning)

So we `pre-tokenize` the corpus. i.e., a coarse-grained tokenization over the corpus that helps us count how often pairs of characters appear.

PAT = r"""'(?:[sdmt]|ll|ve|re)| ?\p{L}+| ?\p{N}+| ?[^\s\p{L}\p{N}]+|\s+(?!\S)|\s+"""

In [ ]:
import regex as re
PAT = r"""'(?:[sdmt]|ll|ve|re)| ?\p{L}+| ?\p{N}+| ?[^\s\p{L}\p{N}]+|\s+(?!\S)|\s+"""
re.findall(PAT, "some text that i'll pre-tokenize")

# In real code, you should use the re.finditer() to avoid storing the pre-tokenized words as
# you construct the mapping from pre-tokens to their counts.

['some', ' text', ' that', ' i', "'ll", ' pre', '-', 'tokenize']